In [ ]:
# --- Cell 1: deps + Drive ---
!pip -q install timm==1.0.7 torchmetrics==1.4.0

from google.colab import drive
drive.mount('/content/drive')

# Your dataset path (as provided)
DATA_ROOT = "/content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset"

import os, sys, pathlib
from pathlib import Path

assert Path(DATA_ROOT).exists(), f"DATA_ROOT not found: {DATA_ROOT}"
print("DATA_ROOT:", DATA_ROOT)

# Quick sanity check of split folders (should contain class subfolders with images)
print("Has train? ", (Path(DATA_ROOT)/"train").exists())
print("Has valid? ", (Path(DATA_ROOT)/"valid").exists())
print("Has test?  ", (Path(DATA_ROOT)/"test").exists())


Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset
Has train?  True
Has valid?  True
Has test?   True


In [ ]:
# --- Cell 2: build/refresh validation split (STRATIFIED) ---
revalidate_script = r"""
import os, shutil, random, argparse
from pathlib import Path
from collections import defaultdict

IMG_EXTS = {".jpg",".jpeg",".png",".webp",".bmp"}

def list_images(dir_path):
    items = []
    for cls in sorted(p.name for p in Path(dir_path).iterdir() if p.is_dir()):
        cdir = Path(dir_path)/cls
        files = [f for f in cdir.rglob("*") if f.suffix.lower() in IMG_EXTS]
        items.append((cls, files))
    return items

def count_split(root, split):
    out = {}
    d = Path(root)/split
    if not d.exists(): return out
    for cls in sorted(p.name for p in d.iterdir() if p.is_dir()):
        n = sum(1 for f in (d/cls).rglob("*") if f.suffix.lower() in IMG_EXTS)
        out[cls] = n
    return out

def pretty_counts(tag, dct):
    total = sum(dct.values())
    parts = " | ".join([f"{k}:{v}" for k,v in dct.items()])
    print(f"{tag} Total={total}  {parts}")

def ensure_dirs(pathes):
    for p in pathes:
        Path(p).mkdir(parents=True, exist_ok=True)

def main(args):
    random.seed(args.seed)
    root = Path(args.data_root)
    train_dir = root/"train"
    valid_dir = root/"valid"

    assert train_dir.exists(), f"Missing {train_dir}"

    tr_counts = count_split(root, "train")
    va_counts = count_split(root, "valid")
    te_counts = count_split(root, "test")
    print("Before:")
    pretty_counts("  train", tr_counts)
    if va_counts: pretty_counts("  valid", va_counts)
    if te_counts: pretty_counts("  test ", te_counts)

    # Move any existing valid back to train (start fresh)
    if valid_dir.exists():
        for cls in sorted(p.name for p in valid_dir.iterdir() if p.is_dir()):
            for f in (valid_dir/cls).rglob("*"):
                if f.is_file() and f.suffix.lower() in IMG_EXTS:
                    dst = train_dir/cls/f.name
                    if dst.exists():
                        stem, suf = dst.stem, dst.suffix
                        k = 1
                        while (train_dir/cls/f"{stem}_{k}{suf}").exists():
                            k += 1
                        dst = train_dir/cls/f"{stem}_{k}{suf}"
                    (train_dir/cls).mkdir(parents=True, exist_ok=True)
                    shutil.move(str(f), str(dst))

    # Re-list after reset
    pool = {}
    for cls, files in list_images(train_dir):
        pool[cls] = files[:]

    # Determine targets (stratified)
    from collections import Counter
    tr_counts_after = {cls: len(files) for cls, files in pool.items()}
    classes = sorted(tr_counts_after.keys())
    target = {}
    for c in classes:
        target[c] = max(1, int(round(args.val_ratio * tr_counts_after[c])))

    print("\nPlanned valid counts:")
    pretty_counts("  target", target)

    # Create valid class dirs
    valid_dir.mkdir(parents=True, exist_ok=True)
    for c in classes:
        (valid_dir/c).mkdir(parents=True, exist_ok=True)

    # Sample & move
    moved = defaultdict(int)
    for c in classes:
        files = pool[c][:]
        random.shuffle(files)
        k = min(target[c], len(files))
        chosen = files[:k]
        for f in chosen:
            dst = valid_dir/c/f.name
            if dst.exists():
                stem, suf = dst.stem, dst.suffix
                i = 1
                while (valid_dir/c/f"{stem}_{i}{suf}").exists():
                    i += 1
                dst = valid_dir/c/f"{stem}_{i}{suf}"
            shutil.move(str(f), str(dst))
            moved[c] += 1

    tr_counts2 = count_split(root, "train")
    va_counts2 = count_split(root, "valid")
    te_counts2 = count_split(root, "test")

    print("\nAfter:")
    pretty_counts("  train", tr_counts2)
    pretty_counts("  valid", va_counts2)
    if te_counts2: pretty_counts("  test ", te_counts2)

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--val_ratio", type=float, default=0.12)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()
    main(args)
"""
with open("revalidate.py","w") as f:
    f.write(revalidate_script)

# Run once (stratified valid = 12% of available train per class)
!python revalidate.py --data_root "$DATA_ROOT" --val_ratio 0.12 --seed 42


Before:
  train Total=1408  Angry:355 | Fear:358 | Happy:345 | Sad:350
  valid Total=192  Angry:48 | Fear:49 | Happy:47 | Sad:48
  test  Total=141  Angry:36 | Fear:35 | Happy:35 | Sad:35

Planned valid counts:
  target Total=192  Angry:48 | Fear:49 | Happy:47 | Sad:48

After:
  train Total=1408  Angry:355 | Fear:358 | Happy:345 | Sad:350
  valid Total=192  Angry:48 | Fear:49 | Happy:47 | Sad:48
  test  Total=141  Angry:36 | Fear:35 | Happy:35 | Sad:35


In [ ]:
# --- Cell 3: training script (ViT with strong regularization) ---
trainer = r"""
import os, math, json, time, argparse, random
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler

import timm
from torchvision import datasets, transforms

try:
    from torchvision.transforms import RandAugment
    HAS_RANDAUG = True
except Exception:
    HAS_RANDAUG = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class MixupCutmix:
    def __init__(self, mixup_alpha=0.4, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5, num_classes=4):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.switch_prob = switch_prob
        self.num_classes = num_classes
        self._eps = 1e-6

    def _one_hot(self, y):
        y = y.view(-1, 1)
        return torch.zeros(y.size(0), self.num_classes, device=y.device).scatter_(1, y, 1.)

    def _rand_bbox(self, W, H, lam):
        cut_rat = math.sqrt(1. - lam)
        cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
        cx = random.randint(0, W - 1)
        cy = random.randint(0, H - 1)
        x1 = max(cx - cut_w // 2, 0)
        y1 = max(cy - cut_h // 2, 0)
        x2 = min(cx + cut_w // 2, W)
        y2 = min(cy + cut_h // 2, H)
        return x1, y1, x2, y2

    def __call__(self, x, y):
        if random.random() > self.prob:
            return x, self._one_hot(y)
        bs = x.size(0)
        perm = torch.randperm(bs, device=x.device)
        x2, y2 = x[perm], y[perm]
        y1_oh = self._one_hot(y)
        y2_oh = self._one_hot(y2)
        use_cutmix = (random.random() < self.switch_prob)
        if use_cutmix and self.cutmix_alpha > 0:
            lam = torch.distributions.Beta(self.cutmix_alpha, self.cutmix_alpha).sample().item()
            W, H = x.size(3), x.size(2)
            x1, y1, x2b, y2b = self._rand_bbox(W, H, lam)
            x[:, :, y1:y2b, x1:x2b] = x2[:, :, y1:y2b, x1:x2b]
            box_area = (x2b - x1) * (y2b - y1)
            lam = 1. - box_area / float(W * H + self._eps)
        else:
            lam = torch.distributions.Beta(self.mixup_alpha, self.mixup_alpha).sample().item()
            x = x * lam + x2 * (1. - lam)
        targets = y1_oh * lam + y2_oh * (1. - lam)
        return x, targets

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.clone().detach()

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.clone()
                p.data.copy_(self.shadow[n].data)

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n].data)
        self.backup = {}

def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def build_transforms(img_size: int):
    train_tf = [
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        transforms.RandomGrayscale(p=0.1),
    ]
    if HAS_RANDAUG:
        train_tf.append(RandAugment(num_ops=2, magnitude=7))
    train_tf += [
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value="random"),
    ]
    eval_tf = transforms.Compose([
        transforms.Resize(int(img_size*1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])
    return transforms.Compose(train_tf), eval_tf

def make_weighted_sampler(dataset):
    ys = [y for _, y in dataset.samples] if hasattr(dataset, "samples") else [y for _, y in dataset.imgs]
    from collections import Counter
    counts = Counter(ys)
    weights = [1.0 / counts[y] for y in ys]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True), dict(counts)

@torch.no_grad()
def evaluate(model, loader, device, tta=False):
    model.eval()
    ce = nn.CrossEntropyLoss()
    correct, total, loss_sum, n_batches = 0, 0, 0.0, 0
    preds_all, targs_all = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if tta:
            logit = model(x)
            logit_flip = model(torch.flip(x, dims=[3]))
            logit = (logit + logit_flip) * 0.5
        else:
            logit = model(x)
        loss = ce(logit, y)
        loss_sum += float(loss.item()); n_batches += 1
        p = logit.argmax(1)
        correct += (p == y).sum().item()
        total   += y.numel()
        preds_all.extend(p.tolist())
        targs_all.extend(y.tolist())
    acc = correct / max(1,total)
    return acc, (loss_sum/max(1,n_batches)), preds_all, targs_all

def train_vit(
    data_root,
    out_dir,
    model_name="vit_base_patch16_224",
    img_size=224,
    epochs=60,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    early_stop=12,
    balance_sampler=True,
    num_workers=2,
    seed=42,
    ema_decay=0.999,
    head_warmup=3,
    grad_clip=1.0,
    mixup_alpha=0.4,
    cutmix_alpha=1.0,
    tta_eval=True,
):
    set_seed(seed)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    train_tf, eval_tf = build_transforms(img_size)

    # dataset
    train_ds = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(data_root, "valid"), transform=eval_tf)
    test_ds  = datasets.ImageFolder(os.path.join(data_root, "test"),  transform=eval_tf)
    class_names = train_ds.classes
    num_classes = len(class_names)
    print("Classes:", class_names)

    if balance_sampler:
        sampler, class_counts = make_weighted_sampler(train_ds)
        print("Using WeightedRandomSampler with class counts:", class_counts)
        train_dl = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=True)
    else:
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_dl  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

    # model with stochastic depth (drop_path)
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes, drop_path_rate=0.1)
    model.to(DEVICE)

    # param groups: gentler LR for backbone
    head, backbone = [], []
    for n, p in model.named_parameters():
        if any(k in n for k in ["head", "classifier", "fc", "pre_logits"]):
            head.append(p)
        else:
            backbone.append(p)

    criterion_hard = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW([
        {"params": backbone, "lr": lr * 0.5},
        {"params": head,     "lr": lr * 1.0},
    ], weight_decay=weight_decay)

    ema = EMA(model, decay=ema_decay)
    mixcut = MixupCutmix(mixup_alpha=mixup_alpha, cutmix_alpha=cutmix_alpha, prob=1.0,
                         switch_prob=0.5, num_classes=num_classes)

    best_val_acc = 0.0
    best_state = None
    no_improve = 0
    warmup_epochs = max(2, min(5, epochs//10))
    history = {"epoch": [], "train_acc": [], "train_loss": [], "val_acc": [], "val_loss": [], "lr": []}

    def set_backbone_requires_grad(flag):
        for n, p in model.named_parameters():
            if p.requires_grad and not any(k in n for k in ["head", "classifier", "fc", "pre_logits"]):
                p.requires_grad = flag

    # freeze backbone for a few epochs (linear-probe warmup)
    set_backbone_requires_grad(False)

    for epoch in range(1, epochs+1):
        # cosine LR with warmup
        if epoch <= warmup_epochs:
            scale = (0.1 + 0.9 * (epoch / warmup_epochs))
        else:
            t = epoch - warmup_epochs
            T = max(1, epochs - warmup_epochs)
            scale = 0.1 + 0.9 * (0.5*(1 + math.cos(math.pi * t / T)))
        for i, g in enumerate(optimizer.param_groups):
            base = lr * (0.5 if i == 0 else 1.0)
            g["lr"] = base * scale

        if epoch == head_warmup + 1:
            set_backbone_requires_grad(True)

        # train
        model.train()
        total, correct = 0, 0
        loss_sum, batches = 0.0, 0
        t0 = time.time()

        for x, y in train_dl:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            x_mix, y_soft = mixcut(x, y)

            optimizer.zero_grad(set_to_none=True)
            logit = model(x_mix)

            log_probs = torch.log_softmax(logit, dim=1)
            loss = -(y_soft * log_probs).sum(dim=1).mean()
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            ema.update(model)

            with torch.no_grad():
                p = logit.argmax(1)
                correct += (p == y).sum().item()
                total   += y.numel()
                loss_sum += float(loss.item())
                batches  += 1

        train_acc  = correct / max(1,total)
        train_loss = loss_sum / max(1,batches)

        # validate with EMA + TTA
        ema.apply_to(model)
        val_acc, val_loss, _, _ = evaluate(model, val_dl, DEVICE, tta=True)
        ema.restore(model)

        history["epoch"].append(epoch)
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss)
        history["lr"].append(optimizer.param_groups[1]["lr"])

        print(f"Epoch {epoch:02d}/{epochs} | train_loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val_loss {val_loss:.4f} acc {val_acc:.4f} | "
              f"lr_head {optimizer.param_groups[1]['lr']:.2e} | "
              f"time {time.time()-t0:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            no_improve = 0
            torch.save(best_state, Path(out_dir) / "best_vit_regularized.pth")
        else:
            no_improve += 1
            if no_improve >= early_stop:
                print(f"Early stopping at epoch {epoch}. Best val acc={best_val_acc:.4f}")
                break

    # save training curve
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(out_dir)/"history.json","w") as f:
        json.dump(history, f, indent=2)

    if best_state is not None:
        model.load_state_dict(best_state)

    # export TorchScript
    model.eval()
    example = torch.randn(1, 3, img_size, img_size).to(DEVICE)
    ts = torch.jit.trace(model.to(DEVICE), example)
    ts_path = str(Path(out_dir) / "best.torchscript")
    ts.save(ts_path)
    print("Saved TorchScript to:", ts_path)

    # final metrics (EMA + TTA)
    ema.apply_to(model)
    tr_acc, tr_loss, _, _ = evaluate(model, train_dl, DEVICE, tta=True)
    va_acc, va_loss, _, _ = evaluate(model, val_dl, DEVICE, tta=True)
    te_acc, te_loss, te_preds, te_targs = evaluate(model, test_dl, DEVICE, tta=True)
    ema.restore(model)

    print("\\nFINAL ACCURACY")
    print(f"  Train: {tr_acc:.4f}")
    print(f"  Valid: {va_acc:.4f}")
    print(f"  Test : {te_acc:.4f}")

    # confusion matrix + per-class acc
    import numpy as np
    cm = np.zeros((len(class_names), len(class_names)), dtype=int)
    for t, p in zip(te_targs, te_preds):
        cm[t, p] += 1
    print("\\nTest Confusion Matrix (rows=true, cols=pred):")
    for r in range(len(class_names)):
        print(class_names[r].ljust(8), cm[r].tolist())
    per_class = (cm.diagonal() / cm.sum(axis=1).clip(min=1)).tolist()
    print("\\nPer-class test accuracy:")
    for c, a in zip(class_names, per_class):
        print(f"  {c}: {a:.3f}")

    with open(Path(out_dir)/"final_metrics.json","w") as f:
        json.dump({
            "classes": class_names,
            "train_acc": tr_acc, "train_loss": tr_loss,
            "val_acc": va_acc,   "val_loss": va_loss,
            "test_acc": te_acc,  "test_loss": te_loss,
            "confusion_matrix": cm.tolist(),
            "per_class_acc": per_class
        }, f, indent=2)

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", default="/content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset")
    ap.add_argument("--out_dir",   default="/content/runs_vit_regularized")
    ap.add_argument("--model_name", default="vit_base_patch16_224")
    ap.add_argument("--img_size", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--weight_decay", type=float, default=1e-4)
    ap.add_argument("--early_stop", type=int, default=12)
    ap.add_argument("--balance_sampler", action="store_true")
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--ema_decay", type=float, default=0.999)
    ap.add_argument("--head_warmup", type=int, default=3)
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--mixup_alpha", type=float, default=0.4)
    ap.add_argument("--cutmix_alpha", type=float, default=1.0)
    ap.add_argument("--tta_eval", action="store_true")
    return ap.parse_args()

if __name__ == "__main__":
    args = parse_args()
    train_vit(
        data_root=args.data_root,
        out_dir=args.out_dir,
        model_name=args.model_name,
        img_size=args.img_size,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        weight_decay=args.weight_decay,
        early_stop=args.early_stop,
        balance_sampler=args.balance_sampler,
        num_workers=args.num_workers,
        seed=args.seed,
        ema_decay=args.ema_decay,
        head_warmup=args.head_warmup,
        grad_clip=args.grad_clip,
        mixup_alpha=args.mixup_alpha,
        cutmix_alpha=args.cutmix_alpha,
        tta_eval=args.tta_eval,
    )
"""
with open("train_vit_regularized.py","w") as f:
    f.write(trainer)

# Train (balanced training sampler ON)
!python train_vit_regularized.py --balance_sampler


Classes: ['Angry', 'Fear', 'Happy', 'Sad']
Using WeightedRandomSampler with class counts: {0: 355, 1: 358, 2: 345, 3: 350}
model.safetensors: 100% 346M/346M [00:04<00:00, 82.5MB/s]
Epoch 01/60 | train_loss 1.4014 acc 0.3075 | val_loss 1.6646 acc 0.2604 | lr_head 8.40e-05 | time 308.2s
Epoch 02/60 | train_loss 1.3333 acc 0.3722 | val_loss 1.5376 acc 0.3229 | lr_head 1.38e-04 | time 95.4s
Epoch 03/60 | train_loss 1.2307 acc 0.3786 | val_loss 1.4259 acc 0.3906 | lr_head 1.92e-04 | time 55.9s
Epoch 04/60 | train_loss 1.2025 acc 0.4304 | val_loss 1.3207 acc 0.4375 | lr_head 2.46e-04 | time 46.1s
Epoch 05/60 | train_loss 1.2044 acc 0.4460 | val_loss 1.2358 acc 0.4896 | lr_head 3.00e-04 | time 44.9s
Epoch 06/60 | train_loss 1.1250 acc 0.4702 | val_loss 1.1615 acc 0.5312 | lr_head 3.00e-04 | time 44.8s
Epoch 07/60 | train_loss 1.1714 acc 0.4446 | val_loss 1.1000 acc 0.5781 | lr_head 2.99e-04 | time 46.9s
Epoch 08/60 | train_loss 1.1444 acc 0.4688 | val_loss 1.0538 acc 0.6146 | lr_head 2.98e-04

In [ ]:
# Cell 1 — deps + paths
!pip -q install scikit-learn==1.5.2 timm==1.0.7

from pathlib import Path
DATA_ROOT = "/content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset"
OUT_DIR   = Path("/content/runs_vit_regularized")  # where training saved files
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 54.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
DATA_ROOT: /content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset
OUT_DIR: /content/runs_vit_regularized


In [ ]:
# Cell 2 — data loaders (same normalization/resize as training eval)
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

eval_tf = transforms.Compose([
    transforms.Resize(int(224*1.15)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

test_ds  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_tf)
val_ds   = datasets.ImageFolder(f"{DATA_ROOT}/valid", transform=eval_tf)
train_ds = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=eval_tf)  # for train metrics if needed

CLASSES = test_ds.classes
NUM_CLASSES = len(CLASSES)
print("Classes:", CLASSES)

BATCH = 32
val_dl  = DataLoader(val_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
train_dl= DataLoader(train_ds,batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)


Classes: ['Angry', 'Fear', 'Happy', 'Sad']


In [ ]:
# Cell 3 — recreate the model, load best weights (FIXED)
import timm, torch, torch.nn as nn
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=NUM_CLASSES, drop_path_rate=0.1)
state = torch.load(OUT_DIR / "best_vit_regularized.pth", map_location="cpu")
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

@torch.no_grad()
def predict_loader(dl, tta=True):
    all_probs, all_preds, all_targs = [], [], []
    ce = nn.CrossEntropyLoss(reduction="sum")
    loss_sum, count = 0.0, 0

    for x, y in dl:
        x = x.to(DEVICE); y = y.to(DEVICE)

        if tta:
            logit = model(x)
            logit_flip = model(torch.flip(x, dims=[3]))
            logit = 0.5 * (logit + logit_flip)
        else:
            logit = model(x)

        loss_sum += ce(logit, y).item()
        count += y.numel()

        prob = torch.softmax(logit, dim=1)
        pred = prob.argmax(1)

        all_probs.append(prob.cpu())
        all_preds.append(pred.cpu())
        all_targs.append(y.cpu())

    return (
        torch.cat(all_probs),
        torch.cat(all_preds),
        torch.cat(all_targs),
        loss_sum / max(1, count),
    )

probs_val, preds_val, y_val, val_loss = predict_loader(val_dl, tta=True)
probs_te,  preds_te,  y_te,  te_loss  = predict_loader(test_dl, tta=True)

print("val_loss:", val_loss, "test_loss:", te_loss)


val_loss: 0.5537398904561996 test_loss: 0.5465744978992652


In [ ]:
!pip -q install opencv-python-headless


In [ ]:
# Cell 4 — full metrics + plots (saved into OUT_DIR)
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def compute_and_save_metrics(y_true, y_pred, probs, split_name):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    p_weight,r_weight,f_weight,_ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n=== {split_name} METRICS ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro   : P={p_macro:.4f} R={r_macro:.4f} F1={f_macro:.4f}")
    print(f"Micro   : P={p_micro:.4f} R={r_micro:.4f} F1={f_micro:.4f}")
    print(f"Weighted: P={p_weight:.4f} R={r_weight:.4f} F1={f_weight:.4f}")

    # per-class report
    report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0)
    (OUT_DIR/f"{split_name}_classification_report.txt").write_text(report)
    print("\nPer-class report saved to:", OUT_DIR/f"{split_name}_classification_report.txt")

    # confusion matrix figure
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig = plt.figure(figsize=(5.5,4.5), dpi=140)
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"{split_name} Confusion Matrix")
    plt.xticks(range(NUM_CLASSES), CLASSES, rotation=30, ha='right')
    plt.yticks(range(NUM_CLASSES), CLASSES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
    fig_path = OUT_DIR/f"{split_name}_confusion_matrix.png"
    plt.savefig(fig_path); plt.close(fig)
    print("Confusion matrix image:", fig_path)

compute_and_save_metrics(y_val.numpy(), preds_val.numpy(), probs_val.numpy(), "valid")
compute_and_save_metrics(y_te.numpy(),  preds_te.numpy(),  probs_te.numpy(),  "test")



=== valid METRICS ===
Accuracy: 0.7969
Macro   : P=0.7966 R=0.7977 F1=0.7960
Micro   : P=0.7969 R=0.7969 F1=0.7969
Weighted: P=0.7963 R=0.7969 F1=0.7954

Per-class report saved to: /content/runs_vit_regularized/valid_classification_report.txt
Confusion matrix image: /content/runs_vit_regularized/valid_confusion_matrix.png

=== test METRICS ===
Accuracy: 0.8369
Macro   : P=0.8369 R=0.8369 F1=0.8364
Micro   : P=0.8369 R=0.8369 F1=0.8369
Weighted: P=0.8366 R=0.8369 F1=0.8362

Per-class report saved to: /content/runs_vit_regularized/test_classification_report.txt
Confusion matrix image: /content/runs_vit_regularized/test_confusion_matrix.png


In [ ]:
# Cell 5 — ViT attention-rollout (no hooks; uses qkv exactly like timm)
import math, random, numpy as np, cv2, torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

model.eval()
DEVICE = next(model.parameters()).device

# Preprocess (same spatial path as eval)
resize_crop = transforms.Compose([
    transforms.Resize(int(224*1.15)),
    transforms.CenterCrop(224),
])
to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

save_dir = OUT_DIR / "viz_attention"
save_dir.mkdir(parents=True, exist_ok=True)

# Group test samples by class (uses test_ds from earlier cells)
random.seed(42)
by_class = {c: [] for c in CLASSES}
for path, label in test_ds.samples:
    by_class[CLASSES[label]].append(path)

def overlay_heatmap_on_img(img_np, heatmap_np, out_path):
    heatmap = (heatmap_np - heatmap_np.min()) / (np.ptp(heatmap_np) + 1e-8)
    heatmap_color = cv2.applyColorMap((heatmap*255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = (0.5*heatmap_color + 0.5*img_np).astype(np.uint8)
    cv2.imwrite(out_path, overlay)

@torch.no_grad()
def attention_rollout_qkv(x, discard_ratio=0.0):
    """
    Compute rollout from each Transformer block's self-attention:
    - at each block: attn = softmax(q @ k^T / sqrt(d))
    - average heads -> [B, T, T]
    - rollout across layers using (I + A) row-normalized
    Returns [B, T] attention from CLS to all tokens.
    """
    # Patch embed + CLS + pos + drop
    B = x.shape[0]
    x_tok = model.patch_embed(x)                       # [B, N, C]
    cls = model.cls_token.expand(B, -1, -1)            # [B, 1, C]
    x_tok = torch.cat((cls, x_tok), dim=1)             # [B, N+1, C]
    x_tok = model.pos_drop(x_tok + model.pos_embed)    # [B, T, C], T=N+1

    attn_per_layer = []
    for blk in model.blocks:
        attn = blk.attn
        # qkv: [B, T, 3*C] -> reshape per timm’s Attention.forward
        qkv = attn.qkv(x_tok)                                                # [B, T, 3*C]
        qkv = qkv.reshape(B, x_tok.shape[1], 3, attn.num_heads, attn.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                  # each [B, heads, T, head_dim]
        # scaled dot-product
        scale = 1.0 / (attn.head_dim ** 0.5)
        a = (q @ k.transpose(-2, -1)) * scale                                 # [B, heads, T, T]
        a = a.softmax(dim=-1)
        # average heads
        a = a.mean(dim=1)                                                     # [B, T, T]
        attn_per_layer.append(a)
        # forward the block normally for the next layer’s qkv
        x_tok = blk(x_tok)

    # rollout
    B, T, _ = attn_per_layer[0].shape
    I = torch.eye(T, device=x_tok.device).unsqueeze(0).expand(B, T, T)
    rollout = None
    for A in attn_per_layer:
        if discard_ratio > 0:
            # optionally prune weakest links (rarely needed)
            flat = A.view(B, -1)
            k = int(flat.shape[1] * discard_ratio)
            if k > 0:
                idx = flat.topk(k, dim=1, largest=False).indices
                flat.scatter_(1, idx, 0.0)
                A = flat.view(B, T, T)
        A = A + I
        A = A / A.sum(dim=-1, keepdim=True)
        rollout = A if rollout is None else rollout @ A

    return rollout[:, 0, :]  # CLS-to-all tokens, shape [B, T]

# Determine patch grid size (e.g., 14x14 for 224/16)
num_patches = model.patch_embed.num_patches  # 196
side = int(math.sqrt(num_patches))           # 14

N_PER_CLASS = 2
saved_paths = []

for cls in CLASSES:
    imgs = by_class[cls]
    random.shuffle(imgs)
    for pth in imgs[:N_PER_CLASS]:
        im = Image.open(pth).convert("RGB")
        im_resized = resize_crop(im)
        x = to_tensor_norm(im_resized).unsqueeze(0).to(DEVICE)

        cls_all = attention_rollout_qkv(x, discard_ratio=0.0)  # [1, T]
        mask_vec = cls_all[0, 1:].detach().cpu().numpy()        # drop CLS -> length = num_patches
        # reshape to patch grid and resize back to image size
        mask_grid = mask_vec.reshape(side, side)
        mask_resized = cv2.resize(mask_grid, (im_resized.width, im_resized.height), interpolation=cv2.INTER_CUBIC)

        # overlay & save
        np_img = np.array(im_resized)[:, :, ::-1]               # BGR for cv2
        out_path = str(save_dir / f"{cls}_{Path(pth).stem}_attn.png")
        overlay_heatmap_on_img(np_img, mask_resized, out_path)
        saved_paths.append(out_path)
        print("Saved:", out_path)

print(f"\nDone. {len(saved_paths)} attention overlays saved in {save_dir}")

Saved: /content/runs_vit_regularized/viz_attention/Angry_a41_jpg.rf.7f9efeac7c72d07265f0642d6a5d0148_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Angry_a4_jpg.rf.84c570868e117f0a947e0c9dda981361_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Fear_f48_jpg.rf.3c8b25393303cd368914efdf3180faba_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Fear_new_f2_png.rf.c9fbc808b4b7f86b96bd2e3a2f82169f_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Happy_h58_jpg.rf.1b2bca429876c66de37cdd2f6a630560_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Happy_new_h15_jpg.rf.cde6689064843993326cd11531fab52b_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Sad_s46_jpeg.rf.6670120bc3f71c84771432e538b4bef7_attn.png
Saved: /content/runs_vit_regularized/viz_attention/Sad_s32_jpg.rf.f334f60bc1625984ffcf5bb90d093120_attn.png

Done. 8 attention overlays saved in /content/runs_vit_regularized/viz_attention


In [ ]:
import json, shutil, os, zipfile
from pathlib import Path

OUT_DIR = Path("/content/runs_vit_regularized")

# Save your classes
classes = ["Angry","Fear","Happy","Sad"]  # update if needed
(OUT_DIR / "classes.json").write_text(json.dumps(classes, indent=2))

# Zip the model and classes
zip_path = "/content/HiddenTales_Model.zip"
with zipfile.ZipFile(zip_path, "w") as zipf:
    zipf.write(OUT_DIR/"best.torchscript", arcname="best.torchscript")
    zipf.write(OUT_DIR/"classes.json", arcname="classes.json")

print("✅ Model package ready at:", zip_path)


✅ Model package ready at: /content/HiddenTales_Model.zip


In [ ]:
from google.colab import files
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === Cell: Upload & Classify one image in Colab ===
# Works whether you loaded:
#  1) the timm model with best_vit_regularized.pth (earlier cells), OR
#  2) TorchScript at OUT_DIR/'best.torchscript'
#
# Output: printed prediction + probs and a preview image with predicted label.

import io, json, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from google.colab import files
from pathlib import Path
from torchvision import transforms

# Paths (must match your earlier cells)
OUT_DIR = Path("/content/runs_vit_regularized")
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

# Classes (try to read from classes.json; fallback to earlier dataset)
try:
    CLASSES = json.loads((OUT_DIR/"classes.json").read_text())
except Exception:
    # fallback to what we used
    CLASSES = ["Angry", "Fear", "Happy", "Sad"]

# Preprocess (same as eval)
preprocess = transforms.Compose([
    transforms.Resize(int(224*1.15)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

# Try to use an already-loaded 'model'. If not present, try to load TorchScript.
try:
    _ = model  # noqa
    model.eval().to(DEVICE)
except NameError:
    print("No 'model' found in memory — loading TorchScript from OUT_DIR.")
    ts_path = OUT_DIR / "best.torchscript"
    assert ts_path.exists(), f"Missing TorchScript at {ts_path}. Re-run training/export."
    model = torch.jit.load(str(ts_path), map_location=DEVICE).eval()

# 1) Upload an image
print("📤 Choose a drawing image to classify (jpg/png)...")
upload = files.upload()  # opens chooser
assert len(upload) > 0, "No file uploaded."
fname = list(upload.keys())[0]
print("Uploaded:", fname)

# 2) Open and preprocess
img = Image.open(io.BytesIO(upload[fname])).convert("RGB")
img_disp = ImageOps.contain(img.copy(), (448, 448))  # for display only (not used in model)
x = preprocess(img).unsqueeze(0).to(DEVICE)

# 3) Predict (with simple TTA: +horizontal flip)
with torch.no_grad():
    logits = model(x)
    logits_flip = model(torch.flip(x, dims=[3]))
    logits = 0.5 * (logits + logits_flip)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

pred_idx = int(np.argmax(probs))
pred_label = CLASSES[pred_idx]

# 4) Show results
print("\n🔮 Prediction:", pred_label)
print("📊 Probabilities:")
for i, p in enumerate(probs):
    print(f"  {CLASSES[i]:>6}: {p:.4f}")

# Small bar chart
plt.figure(figsize=(4.8, 3))
plt.bar(CLASSES, probs)
plt.title(f"Prediction: {pred_label}")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Display the uploaded image
plt.figure(figsize=(4.8, 4.8))
plt.imshow(img_disp)
plt.axis("off")
plt.title(f"Predicted: {pred_label}")
plt.show()


In [ ]:
# ============================
# METRICS: Train / Valid / Test + Confusion Matrices (Colab-ready)
# ============================
import os, json, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

# ---- Update paths here if needed ----
DATA_ROOT = "/content/drive/MyDrive/datasetLatestFYP-20251024T150709Z-1-001/datasetLatestFYP/labeldataset"
OUT_DIR   = Path("/content/runs_vit_regularized")
MODEL_PTH = OUT_DIR / "best_vit_regularized.pth"  # saved by your training script

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
BATCH = 32
NUM_WORKERS = 2

assert Path(DATA_ROOT).exists(), f"Not found: {DATA_ROOT}"
assert MODEL_PTH.exists(), f"Missing model weights: {MODEL_PTH} (re-run training cell first)"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Eval transforms (must match training eval) ----
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

# ---- Datasets & loaders (clean eval; no Mixup/CutMix) ----
train_ds = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=eval_tf)
val_ds   = datasets.ImageFolder(f"{DATA_ROOT}/valid", transform=eval_tf)
test_ds  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_tf)
CLASSES  = train_ds.classes
NCLS     = len(CLASSES)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ---- Rebuild model & load best weights ----
import timm
model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=NCLS, drop_path_rate=0.1)
state = torch.load(MODEL_PTH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

@torch.no_grad()
def predict_loader(dl, tta=True):
    """Return probs, preds, labels, avg loss (CE)."""
    ce = nn.CrossEntropyLoss(reduction="sum")
    all_probs, all_preds, all_targs = [], [], []
    loss_sum, count = 0.0, 0
    for x, y in dl:
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = model(x)
        if tta:
            logits_flip = model(torch.flip(x, dims=[3]))
            logits = 0.5 * (logits + logits_flip)
        loss_sum += ce(logits, y).item()
        count += y.numel()
        prob = torch.softmax(logits, dim=1)
        pred = prob.argmax(1)
        all_probs.append(prob.cpu()); all_preds.append(pred.cpu()); all_targs.append(y.cpu())
    return (
        torch.cat(all_probs).numpy(),
        torch.cat(all_preds).numpy(),
        torch.cat(all_targs).numpy(),
        loss_sum / max(1, count),
    )

def compute_and_save_metrics(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n=== {split_name.upper()} METRICS ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro   : P={p_macro:.4f} R={r_macro:.4f} F1={f_macro:.4f}")
    print(f"Micro   : P={p_micro:.4f} R={r_micro:.4f} F1={f_micro:.4f}")
    print(f"Weighted: P={p_w:.4f} R={r_w:.4f} F1={f_w:.4f}")

    # Per-class report
    rep_txt = classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0)
    (OUT_DIR/f"{split_name}_classification_report.txt").write_text(rep_txt)
    print("Per-class report saved to:", OUT_DIR/f"{split_name}_classification_report.txt")

    # Confusion matrices (raw + normalized)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NCLS)))

    def _plot_cm(cm_arr, title, fname, normalize=False):
        if normalize:
            cm_sum = cm_arr.sum(axis=1, keepdims=True)
            cm_disp = np.divide(cm_arr, cm_sum, out=np.zeros_like(cm_arr, dtype=float), where=cm_sum!=0)
            fmt = ".2f"
        else:
            cm_disp = cm_arr
            fmt = "d"
        plt.figure(figsize=(6,5), dpi=140)
        plt.imshow(cm_disp, interpolation="nearest")
        plt.title(title)
        plt.xticks(range(NCLS), CLASSES, rotation=30, ha="right")
        plt.yticks(range(NCLS), CLASSES)
        thresh = cm_disp.max()/2 if cm_disp.size else 0
        for i in range(NCLS):
            for j in range(NCLS):
                plt.text(j, i, format(cm_disp[i, j], fmt),
                         ha="center", va="center",
                         color="white" if cm_disp[i, j] > thresh else "black")
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
        plt.savefig(fname); plt.close()
        print("Saved:", fname)

    _plot_cm(cm, f"{split_name.title()} Confusion Matrix", OUT_DIR/f"{split_name}_confusion_matrix_raw.png", normalize=False)
    _plot_cm(cm, f"{split_name.title()} Confusion Matrix (Normalized)", OUT_DIR/f"{split_name}_confusion_matrix_norm.png", normalize=True)

# ---------- RUN (with TTA) ----------
probs_tr, preds_tr, y_tr, tr_loss = predict_loader(train_dl, tta=True)
probs_va, preds_va, y_va, va_loss = predict_loader(val_dl,   tta=True)
probs_te, preds_te, y_te, te_loss = predict_loader(test_dl,  tta=True)

# Print headline accuracies (easy to copy for slides)
acc_tr = accuracy_score(y_tr, preds_tr)
acc_va = accuracy_score(y_va, preds_va)
acc_te = accuracy_score(y_te, preds_te)
print("\n===== FINAL ACCURACY (with TTA) =====")
print(f"Train: {acc_tr:.4f}")
print(f"Valid: {acc_va:.4f}")
print(f"Test : {acc_te:.4f}")

# Full metrics + confusion matrices
compute_and_save_metrics(y_tr, preds_tr, "train")
compute_and_save_metrics(y_va, preds_va, "valid")
compute_and_save_metrics(y_te, preds_te, "test")

# Optional: per-class accuracy on TEST for quick view
cm_test = confusion_matrix(y_te, preds_te, labels=list(range(NCLS)))
per_class_acc = (cm_test.diagonal() / cm_test.sum(axis=1).clip(min=1)).tolist()
print("\nPer-class Test Accuracy:")
for cls, a in zip(CLASSES, per_class_acc):
    print(f"  {cls:<6}: {a:.3f}")

# Save a small JSON summary (useful for your report)
summary = {
    "classes": CLASSES,
    "train": {"acc": float(acc_tr), "loss": float(tr_loss)},
    "valid": {"acc": float(acc_va), "loss": float(va_loss)},
    "test":  {"acc": float(acc_te), "loss": float(te_loss)},
    "per_class_test_acc": {c: float(a) for c, a in zip(CLASSES, per_class_acc)},
}
(OUT_DIR/"metrics_summary.json").write_text(json.dumps(summary, indent=2))
print("\nSaved summary to:", OUT_DIR/"metrics_summary.json")
print("Confusion matrices + reports saved in:", OUT_DIR)



===== FINAL ACCURACY (with TTA) =====
Train: 0.9993
Valid: 0.7969
Test : 0.8369

=== TRAIN METRICS ===
Accuracy: 0.9993
Macro   : P=0.9993 R=0.9993 F1=0.9993
Micro   : P=0.9993 R=0.9993 F1=0.9993
Weighted: P=0.9993 R=0.9993 F1=0.9993
Per-class report saved to: /content/runs_vit_regularized/train_classification_report.txt
Saved: /content/runs_vit_regularized/train_confusion_matrix_raw.png
Saved: /content/runs_vit_regularized/train_confusion_matrix_norm.png

=== VALID METRICS ===
Accuracy: 0.7969
Macro   : P=0.7966 R=0.7977 F1=0.7960
Micro   : P=0.7969 R=0.7969 F1=0.7969
Weighted: P=0.7963 R=0.7969 F1=0.7954
Per-class report saved to: /content/runs_vit_regularized/valid_classification_report.txt
Saved: /content/runs_vit_regularized/valid_confusion_matrix_raw.png
Saved: /content/runs_vit_regularized/valid_confusion_matrix_norm.png

=== TEST METRICS ===
Accuracy: 0.8369
Macro   : P=0.8369 R=0.8369 F1=0.8364
Micro   : P=0.8369 R=0.8369 F1=0.8369
Weighted: P=0.8366 R=0.8369 F1=0.8362
Per-cl

In [ ]:
# ============================
# VISUALIZATIONS for the report (FIXED)
# ============================
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.metrics import precision_recall_fscore_support

# Reuse paths/vars from earlier cells
OUT_DIR = Path("/content/runs_vit_regularized")

# Helper to save+close a single-plot figure
def _save_show(fig, path):
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

# --- 1) Training curves from history.json ---
hist_path = OUT_DIR / "history.json"
if hist_path.exists():
    H = json.loads(hist_path.read_text())
    epochs = H.get("epoch", list(range(1, len(H.get("val_acc", [])) + 1)))

    # Accuracy curve
    fig = plt.figure(figsize=(7, 4.5))
    plt.plot(epochs, H["train_acc"], label="Train Acc")
    plt.plot(epochs, H["val_acc"],   label="Valid Acc")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    _save_show(fig, OUT_DIR / "curve_accuracy.png")

    # Loss curve
    fig = plt.figure(figsize=(7, 4.5))
    plt.plot(epochs, H["train_loss"], label="Train Loss")
    plt.plot(epochs, H["val_loss"],   label="Valid Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    _save_show(fig, OUT_DIR / "curve_loss.png")
else:
    print("history.json not found — skipping training curves.")

# --- 2) Ensure TEST probs/labels exist; if not, recompute quickly using predict_loader ---
need_recompute = any(v not in globals() for v in ("probs_te", "y_te", "preds_te"))
if need_recompute:
    print("Recomputing TEST probabilities (using predict_loader)...")
    # Requires: model, test_dl, predict_loader() from your metrics cell
    probs_te, preds_te, y_te, _ = predict_loader(test_dl, tta=True)

# Convert to numpy (safety)
probs_te = np.asarray(probs_te)
y_te     = np.asarray(y_te)
preds_te = np.asarray(preds_te)

# CLASSES should be defined in your metrics cell
C = len(CLASSES)

# --- 3) Per-class Precision / Recall / F1 bars (TEST) ---
p_cls, r_cls, f_cls, support = precision_recall_fscore_support(
    y_te, preds_te, labels=list(range(C)), average=None, zero_division=0
)

def _bar(values, title, ylabel, fname, ylim=(0,1)):
    fig = plt.figure(figsize=(7, 4.5))
    plt.bar(CLASSES, values)
    if ylim: plt.ylim(*ylim)
    plt.ylabel(ylabel); plt.title(title)
    _save_show(fig, OUT_DIR / fname)

_bar(p_cls, "Per-class Precision (TEST)", "Precision", "test_per_class_precision.png")
_bar(r_cls, "Per-class Recall (TEST)",    "Recall",    "test_per_class_recall.png")
_bar(f_cls, "Per-class F1-score (TEST)",  "F1-score",  "test_per_class_f1.png", ylim=(0,1))
_bar(support, "Per-class Support (TEST)", "Number of Samples", "test_per_class_support.png", ylim=None)

# --- 4) ROC Curves (one-vs-rest), micro & macro (TEST) ---
# Binarize labels
Y_bin = label_binarize(y_te, classes=list(range(C)))  # shape [N, C]

# Ensure P are probabilities for each class
P = probs_te if probs_te.ndim == 2 else np.eye(C)[preds_te]

fpr, tpr, roc_auc = {}, {}, {}
for i in range(C):
    fpr[i], tpr[i], _ = roc_curve(Y_bin[:, i], P[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average
fpr["micro"], tpr["micro"], _ = roc_curve(Y_bin.ravel(), P.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Macro-average (fixed loop)
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(C)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(C):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i], left=0, right=1)
mean_tpr /= max(1, C)
roc_auc["macro"] = auc(all_fpr, mean_tpr)

# Plot ROC (single figure)
fig = plt.figure(figsize=(7, 5))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
plt.plot(all_fpr, mean_tpr,       label=f"macro-average (AUC = {roc_auc['macro']:.3f})")
for i in range(C):
    plt.plot(fpr[i], tpr[i], label=f"{CLASSES[i]} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves (TEST)")
plt.legend(loc="lower right", ncol=2)
_save_show(fig, OUT_DIR / "test_roc_curves.png")

# --- 5) Precision–Recall Curves (one-vs-rest), micro & macro (TEST) ---
precision, recall, ap = {}, {}, {}
for i in range(C):
    precision[i], recall[i], _ = precision_recall_curve(Y_bin[:, i], P[:, i])
    ap[i] = average_precision_score(Y_bin[:, i], P[:, i])

# Micro-average
precision["micro"], recall["micro"], _ = precision_recall_curve(Y_bin.ravel(), P.ravel())
ap["micro"] = average_precision_score(Y_bin, P, average="micro")

# Macro-average (fixed loop)
all_recalls = np.unique(np.concatenate([recall[i] for i in range(C)]))
mean_precision = np.zeros_like(all_recalls)
for i in range(C):
    # interpolate precision as a function of recall
    mean_precision += np.interp(all_recalls, recall[i][::-1], precision[i][::-1], left=0, right=1)
mean_precision /= max(1, C)
ap["macro"] = np.trapz(mean_precision, all_recalls)

# Plot PR (single figure)
fig = plt.figure(figsize=(7, 5))
plt.plot(recall["micro"], precision["micro"], label=f"micro-average (AP = {ap['micro']:.3f})")
plt.plot(all_recalls,  mean_precision,       label=f"macro-average (AP = {ap['macro']:.3f})")
for i in range(C):
    plt.plot(recall[i], precision[i], label=f"{CLASSES[i]} (AP = {ap[i]:.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision–Recall Curves (TEST)")
plt.legend(loc="lower left", ncol=2)
_save_show(fig, OUT_DIR / "test_pr_curves.png")

print("\n✅ Visualization files are ready in:", OUT_DIR)


Saved: /content/runs_vit_regularized/curve_accuracy.png
Saved: /content/runs_vit_regularized/curve_loss.png
Saved: /content/runs_vit_regularized/test_per_class_precision.png
Saved: /content/runs_vit_regularized/test_per_class_recall.png
Saved: /content/runs_vit_regularized/test_per_class_f1.png
Saved: /content/runs_vit_regularized/test_per_class_support.png
Saved: /content/runs_vit_regularized/test_roc_curves.png


/tmp/ipython-input-480231753.py:130: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap["macro"] = np.trapz(mean_precision, all_recalls)


Saved: /content/runs_vit_regularized/test_pr_curves.png

✅ Visualization files are ready in: /content/runs_vit_regularized


In [ ]:
# 1) Zip the whole folder
!zip -r -q /content/runs_vit_regularized.zip /content/runs_vit_regularized

# 2) Download the zip
from google.colab import files
files.download('/content/runs_vit_regularized.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>